In [0]:
CREATE WIDGET TEXT end_date DEFAULT '2025-11-30';

In [0]:
WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

-- ----------------------------------------------------------
-- Claims universes (5y, 3y) Dx + Tx, with NPIs
-- ----------------------------------------------------------
cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant',
        'Anesthesiology',
        'Dentist',
        'Dietitian, Registered',
        'Emergency Medical Technician, Basic',
        'Emergency Medicine',
        'General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered',
        'Obstetrics & Gynecology',
        'Pathology',
        'Radiology',
        'Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry',
        'Psychiatry',
        'Adolescent Medicine',
        'Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine',
        'Nutrition, Pediatric',
        'Oncology, Pediatrics',
        'Pediatric Cardiology',
        'Pediatric Critical Care Medicine',
        'Pediatric Dermatology',
        'Pediatric Emergency Medicine',
        'Pediatric Endocrinology',
        'Pediatric Gastroenterology',
        'Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases',
        'Pediatric Nephrology',
        'Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery',
        'Pediatric Otolaryngology',
        'Pediatric Pulmonology',
        'Pediatric Radiology',
        'Pediatric Rehabilitation Medicine',
        'Pediatric Rheumatology',
        'Pediatric Surgery',
        'Pediatrics',
        'Clinical Biochemical Genetics',
        'Clinical Genetics (M.D.)',
        'Clinical Molecular Genetics',
        'Ph.D. Medical Genetics',
        'Neurodevelopmental Disabilities',
        'Neurology',
        'Neurology with Special Qualifications in Child Neurology',
        'Neuroradiology'
      )
    )
),

all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) or npi is null
),

all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) or npi is null
),

all_claims_5yr AS (
    SELECT DISTINCT * FROM all_dx_claims_5yr
    UNION
    SELECT DISTINCT * FROM all_tx_claims_5yr
),

all_claims_3yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) or npi is null
)

select * from all_tx_claims_5yr
where patient_id in ('07F040KL', '9T9D5NWE', 'Q3TSJ19Q', 'HFDG1Z0F', '6513VL4S', 'E79M532Q', '2W1LYFK2', 'CKWMJ63M')